# Notebook 3: Regression (AKT1 pIC50) -- v2, aligned with the Notebook 2 classifier

This is a rewrite of the regression notebook to match the methodology. The original regression notebook trained
Ridge -> Random Forest -> XGBoost -> ensemble/stacking directly on the full
~2990-column feature matrix, with no feature selection, a 3-fold/40-trial
Optuna budget, and no refit on train+validation before the final test
evaluation. The classifier notebook used 5-fold CV, 100 trials,
**per-fold nested feature selection**, and refit its final model on
train+validation combined before touching the test set. This rewrite ports
all of that over.



# Environment Setup

In [ ]:
!pip -q install optuna xgboost lightgbm shap
!pip install optuna

In [ ]:

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import multiprocessing as mp

from scipy.stats import pearsonr, wilcoxon, ttest_rel
from sklearn.base import clone, BaseEstimator, RegressorMixin
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold, KFold, cross_val_predict
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

from xgboost import XGBRegressor
import xgboost
from lightgbm import LGBMRegressor
import lightgbm
import sklearn
import optuna
import shap
import joblib
from tqdm import tqdm

RANDOM_STATE = 42

TUNE_N_FOLDS = 5
TUNE_N_TRIALS = 100
TRIAL_TIMEOUT = 300
EARLY_STOPPING_PATIENCE = 3

FEATURE_COUNT_CHOICES = [3000, 1500, 750, 400, 200, 100]

N_YRAND_PERMUTATIONS = 200
N_BOOTSTRAP = 1000
WELL_PREDICTED_ERROR_THRESHOLD = 0.5  # pIC50 units; used only for the CSV export at the end

optuna.logging.set_verbosity(optuna.logging.WARNING)

print(f"scikit-learn: {sklearn.__version__} | XGBoost: {xgboost.__version__} | "
      f"LightGBM: {lightgbm.__version__} | Optuna: {optuna.__version__} | SHAP: {shap.__version__}")

plt.rcParams.update({
    "figure.dpi": 140, "savefig.dpi": 300,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "font.size": 10,
})



# Load `features.pkl` and `splits.pkl`

In [ ]:
features = joblib.load("features.pkl")
splits = joblib.load("splits.pkl")

const = features["constants"]
TIER_TO_LABEL = const["TIER_TO_LABEL"]
LABEL_TO_TIER = const["LABEL_TO_TIER"]
ACTIVE_PIC50_CUTOFF = const["ACTIVE_PIC50_CUTOFF"]
INACTIVE_PIC50_CUTOFF = const["INACTIVE_PIC50_CUTOFF"]
AD_TANIMOTO_THRESHOLD = const["AD_TANIMOTO_THRESHOLD"]
AD_KNN_K = const["AD_KNN_K"]

hybrid_fp = features["hybrid_fp"]
desc_df_all = features["desc_2d"]
desc_cols = features["desc_cols"]
y_class_full = features["y_class"]
y_pic50_full = features["pIC50"]
smiles_all = features["smiles"]
scaffold_all = features["scaffold"]
tier_all = features["bioactivity_tier"]

X_train, X_val, X_test = splits["X_train"], splits["X_val"], splits["X_test"]
y_train_pic50, y_val_pic50, y_test_pic50 = splits["y_train_pic50"], splits["y_val_pic50"], splits["y_test_pic50"]
train_idx, val_idx, test_idx = splits["train_idx"], splits["val_idx"], splits["test_idx"]

print("X_train / X_val / X_test:", X_train.shape, X_val.shape, X_test.shape)
print(f"pIC50 range (train): {y_train_pic50.min():.2f} - {y_train_pic50.max():.2f}, "
      f"mean={y_train_pic50.mean():.2f}, std={y_train_pic50.std():.2f}")


# Shared Helpers

As in the classifier notebook, the applicability-domain fingerprint prep is
built later in the AD section, once we know which indices the final model
actually trained on.

In [ ]:
def prepare_filtered_split(train_idx_s, val_idx_s, test_idx_s, variance_threshold=0.01, corr_threshold=0.95):
    fp_cols_s = [f"FP_{i}" for i in range(hybrid_fp.shape[1])]
    X_fp = pd.DataFrame(hybrid_fp, columns=fp_cols_s)
    X_all_s = pd.concat([X_fp, desc_df_all.reset_index(drop=True)], axis=1)

    X_train_raw = X_all_s.iloc[train_idx_s].copy()
    X_val_raw = X_all_s.iloc[val_idx_s].copy()
    X_test_raw = X_all_s.iloc[test_idx_s].copy()

    medians = X_train_raw[desc_cols].median(numeric_only=True).fillna(0.0)
    X_train_raw[desc_cols] = X_train_raw[desc_cols].fillna(medians)
    X_val_raw[desc_cols] = X_val_raw[desc_cols].fillna(medians)
    X_test_raw[desc_cols] = X_test_raw[desc_cols].fillna(medians)

    selector = VarianceThreshold(threshold=variance_threshold)
    X_train_var = pd.DataFrame(selector.fit_transform(X_train_raw), columns=X_train_raw.columns[selector.get_support()])
    X_val_var = pd.DataFrame(selector.transform(X_val_raw), columns=X_train_var.columns)
    X_test_var = pd.DataFrame(selector.transform(X_test_raw), columns=X_train_var.columns)

    corr_matrix = X_train_var.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [c for c in upper.columns if any(upper[c] > corr_threshold)]

    return X_train_var.drop(columns=to_drop), X_val_var.drop(columns=to_drop), X_test_var.drop(columns=to_drop)


def nearest_training_tanimoto(query_fps, train_fps, batch_size=500):
    query_bool = np.asarray(query_fps).astype(np.int32)
    train_bool = np.asarray(train_fps).astype(np.int32)
    train_popcount = train_bool.sum(axis=1)
    nearest = np.empty(len(query_bool), dtype=float)
    for start in range(0, len(query_bool), batch_size):
        batch = query_bool[start:start + batch_size]
        query_popcount = batch.sum(axis=1)
        intersection = batch @ train_bool.T
        union = query_popcount[:, None] + train_popcount[None, :] - intersection
        similarity = np.divide(intersection, union, out=np.zeros_like(intersection, dtype=float), where=union != 0)
        nearest[start:start + batch_size] = similarity.max(axis=1)
    return nearest


def topk_mean_training_tanimoto(query_fps, train_fps, k=5, batch_size=500):
    query_bool = np.asarray(query_fps).astype(np.int32)
    train_bool = np.asarray(train_fps).astype(np.int32)
    train_popcount = train_bool.sum(axis=1)
    k_eff = min(k, len(train_bool))
    topk_mean = np.empty(len(query_bool), dtype=float)
    for start in range(0, len(query_bool), batch_size):
        batch = query_bool[start:start + batch_size]
        query_popcount = batch.sum(axis=1)
        intersection = batch @ train_bool.T
        union = query_popcount[:, None] + train_popcount[None, :] - intersection
        similarity = np.divide(intersection, union, out=np.zeros_like(intersection, dtype=float), where=union != 0)
        top_k_vals = -np.partition(-similarity, k_eff - 1, axis=1)[:, :k_eff]
        topk_mean[start:start + batch_size] = top_k_vals.mean(axis=1)
    return topk_mean


# Baseline: Ridge Regression

In [ ]:
ridge_alpha_grid = np.logspace(-2, 6, 60)
ridge_cv_splits = list(GroupKFold(n_splits=5).split(X_train, y_train_pic50, groups=scaffold_all[train_idx]))


ridge_baseline = make_pipeline(
    StandardScaler(),
    RidgeCV(alphas=ridge_alpha_grid, cv=ridge_cv_splits, scoring="r2"),
)
ridge_baseline.fit(X_train, y_train_pic50)

ridge_chosen_alpha = ridge_baseline.named_steps["ridgecv"].alpha_
print(f"RidgeCV selected alpha: {ridge_chosen_alpha:.4g} "
      f"(searched {len(ridge_alpha_grid)} values, 5-fold scaffold-grouped CV)")

ridge_val_pred = ridge_baseline.predict(X_val)
ridge_test_pred = ridge_baseline.predict(X_test)

ridge_metrics = {
    "R2": r2_score(y_test_pic50, ridge_test_pred),
    "RMSE": np.sqrt(mean_squared_error(y_test_pic50, ridge_test_pred)),
    "MAE": mean_absolute_error(y_test_pic50, ridge_test_pred),
}
print("\nRidge Regression test metrics:")
for k, v in ridge_metrics.items():
    print(f"{k:6s}: {v:.4f}")


# Scaffold-Grouped CV Folds (5-fold) for Optuna

Same 5-fold scaffold-grouped design as the classifier notebook (was 3-fold
here originally). `tune_idx` covers train+validation only; the test set is
never touched during tuning.

In [ ]:
tune_idx = np.concatenate([train_idx, val_idx])
tune_scaffold_groups = scaffold_all[tune_idx]

gkf = GroupKFold(n_splits=TUNE_N_FOLDS)
cv_folds = []
for fold_i, (fold_train_pos, fold_val_pos) in enumerate(gkf.split(tune_idx, groups=tune_scaffold_groups)):
    fold_train_idx = tune_idx[fold_train_pos]
    fold_val_idx = tune_idx[fold_val_pos]
    X_ft, X_fv, _ = prepare_filtered_split(fold_train_idx, fold_val_idx, fold_val_idx)

    cv_folds.append({
        "X_train": X_ft, "X_val": X_fv,
        "y_train_pic50": y_pic50_full[fold_train_idx],
        "y_val_pic50": y_pic50_full[fold_val_idx],
    })
    n_shared = len(set(scaffold_all[fold_train_idx]) & set(scaffold_all[fold_val_idx]))
    print(f"  Fold {fold_i}: {len(fold_train_idx)} train / {len(fold_val_idx)} val, "
          f"{X_ft.shape[1]} features, {n_shared} scaffolds shared (should be 0)")

print(f"\nBuilt {TUNE_N_FOLDS} scaffold-grouped CV folds. Test set ({len(test_idx)}) untouched.")


# Nested Per-Fold Feature Ranking


In [ ]:
def _rank_fold_features_reg(X_tr, y_tr):
    scout = RandomForestRegressor(
        n_estimators=300, max_depth=None, random_state=RANDOM_STATE, n_jobs=-1,
    )
    scout.fit(X_tr, y_tr)
    order = np.argsort(scout.feature_importances_)[::-1]
    return list(X_tr.columns[order])

for fold in cv_folds:
    fold["feature_rank"] = _rank_fold_features_reg(fold["X_train"], fold["y_train_pic50"])

print("Per-fold feature rankings computed (train-partition only, no leakage into fold validation).")
for i, fold in enumerate(cv_folds):
    print(f"  Fold {i}: {len(fold['feature_rank'])} candidate features ranked")

def top_fold_cols(fold, n_features):
    n_take = min(n_features, len(fold["feature_rank"]))
    return fold["feature_rank"][:n_take]


# Random Forest (Optuna-Tuned, nested feature selection)

Same timeout-guarded trial pattern as the classifier: each trial runs in a
subprocess with a hard `TRIAL_TIMEOUT`, so one slow RF configuration can't
stall the whole study.

In [ ]:
RF_N_TRIALS = 100
RF_TRIAL_TIMEOUT = 300
RF_EARLY_STOPPING_PATIENCE = 10
RF_FEATURE_COUNT_CHOICES = [1500, 1000, 750, 500, 300, 150]
RF_FINAL_N_ESTIMATORS_CAP = 1500

def fit_and_score_rf_reg(params):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model_params = {k: v for k, v in params.items() if k != "n_features"}
        n_features = params["n_features"]
        fold_scores = []
        for fold in tqdm(cv_folds, desc="  folds", leave=False):
            cols = top_fold_cols(fold, n_features)
            model = RandomForestRegressor(**model_params, random_state=RANDOM_STATE, n_jobs=-1)
            model.fit(fold["X_train"][cols], fold["y_train_pic50"])
            fold_scores.append(r2_score(fold["y_val_pic50"], model.predict(fold["X_val"][cols])))
    return float(np.mean(fold_scores))

def objective_rf_reg(trial):
    bootstrap = trial.suggest_categorical("bootstrap", [True, False])
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 150, 700),   # was 200-2500
        "max_depth": trial.suggest_int("max_depth", 3, 40),            # was 3-90
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 30),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 15),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", 0.3, 0.5, None]),
        "bootstrap": bootstrap,
        "n_features": trial.suggest_categorical("n_features", RF_FEATURE_COUNT_CHOICES),
    }
    if bootstrap:
        params["max_samples"] = trial.suggest_float("max_samples", 0.5, 1.0)
    with mp.Pool(1) as pool:
        result = pool.apply_async(fit_and_score_rf_reg, (params,))
        try:
            return result.get(timeout=RF_TRIAL_TIMEOUT)
        except Exception:
            return -999.0

study_rf_reg = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))

print(f"Optimizing Random Forest with 5-fold CV (nested feature selection), "
      f"{RF_N_TRIALS} trials, trimmed search space...")
with tqdm(total=RF_N_TRIALS, desc="RF trials") as pbar:
    def callback(study, trial):
        pbar.update(1)
        pbar.set_postfix({
            'Best': f'{study.best_value:.4f}',
            'Current': f'{trial.value:.4f}',
            'Depth': trial.params.get('max_depth', 0),
            'Feats': trial.params.get('n_features', 0),
        })
        if study.best_trial and (trial.number - study.best_trial.number) > RF_EARLY_STOPPING_PATIENCE:
            study.stop()
    study_rf_reg.optimize(objective_rf_reg, n_trials=RF_N_TRIALS, callbacks=[callback])

print("Best RF params:", study_rf_reg.best_params)
print("Best RF CV-mean R2:", study_rf_reg.best_value)

rf_best_params = {k: v for k, v in study_rf_reg.best_params.items() if k != "n_features"}
rf_n_features = study_rf_reg.best_params["n_features"]

tuned_n_estimators = rf_best_params["n_estimators"]
rf_best_params["n_estimators"] = min(tuned_n_estimators * 2, RF_FINAL_N_ESTIMATORS_CAP)
print(f"Final n_estimators boosted from {tuned_n_estimators} -> {rf_best_params['n_estimators']}")

rf_full_rank = _rank_fold_features_reg(X_train, y_train_pic50)
rf_selected_cols = rf_full_rank[:min(rf_n_features, len(rf_full_rank))]
print(f"RF selected feature count: {len(rf_selected_cols)} (Optuna-tuned n_features={rf_n_features})")

best_rf_reg = RandomForestRegressor(**rf_best_params, random_state=RANDOM_STATE, n_jobs=-1)
best_rf_reg.fit(X_train[rf_selected_cols], y_train_pic50)

rf_val_pred = best_rf_reg.predict(X_val[rf_selected_cols])
rf_test_pred = best_rf_reg.predict(X_test[rf_selected_cols])

rf_metrics = {
    "R2": r2_score(y_test_pic50, rf_test_pred),
    "RMSE": np.sqrt(mean_squared_error(y_test_pic50, rf_test_pred)),
    "MAE": mean_absolute_error(y_test_pic50, rf_test_pred),
}
print("\nRandom Forest test metrics:")
for k, v in rf_metrics.items():
    print(f"{k:6s}: {v:.4f}")

# XGBoost (Optuna-Tuned, nested feature selection, early stopping)



In [ ]:

XGB_N_TRIALS = 100
XGB_EARLY_STOPPING_PATIENCE = 10
XGB_FEATURE_COUNT_CHOICES = [2500, 2000, 1750, 1500, 1250, 1000, 750]

def objective_xgb_reg(trial):
    grow_policy = trial.suggest_categorical("grow_policy", ["depthwise", "lossguide"])
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 3000),
        "max_depth": trial.suggest_int("max_depth", 2, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.5, 1.0),
        "min_child_weight": trial.suggest_float("min_child_weight", 1, 20),
        "gamma": trial.suggest_float("gamma", 0, 10),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 20.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 20.0, log=True),
        "grow_policy": grow_policy,
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "tree_method": "hist",
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
        "early_stopping_rounds": 50,
    }
    if grow_policy == "lossguide":
        params["max_leaves"] = trial.suggest_int("max_leaves", 16, 256)

    n_features = trial.suggest_categorical("n_features", XGB_FEATURE_COUNT_CHOICES)
    fold_scores = []
    for fold in tqdm(cv_folds, desc="  folds", leave=False):
        cols = top_fold_cols(fold, n_features)
        model = XGBRegressor(**params)
        model.fit(
            fold["X_train"][cols], fold["y_train_pic50"],
            eval_set=[(fold["X_val"][cols], fold["y_val_pic50"])],
            verbose=False,
        )
        fold_scores.append(r2_score(fold["y_val_pic50"], model.predict(fold["X_val"][cols])))
    return float(np.mean(fold_scores))

study_xgb_reg = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))

print(f"Optimizing XGBoost with 5-fold CV (nested feature selection, early stopping), "
      f"{XGB_N_TRIALS} trials, widened search space...")
with tqdm(total=XGB_N_TRIALS, desc="XGB trials") as pbar:
    def callback(study, trial):
        pbar.update(1)
        pbar.set_postfix({
            'Best': f'{study.best_value:.4f}',
            'Current': f'{trial.value:.4f}',
            'Depth': trial.params.get('max_depth', 0),
            'Feats': trial.params.get('n_features', 0),
        })
        if study.best_trial and (trial.number - study.best_trial.number) > XGB_EARLY_STOPPING_PATIENCE:
            study.stop()
    study_xgb_reg.optimize(objective_xgb_reg, n_trials=XGB_N_TRIALS, callbacks=[callback])

print("Best XGB params:", study_xgb_reg.best_params)
print("Best XGB CV-mean R2:", study_xgb_reg.best_value)

xgb_best_params = {k: v for k, v in study_xgb_reg.best_params.items() if k != "n_features"}
xgb_n_features = study_xgb_reg.best_params["n_features"]

xgb_full_rank = _rank_fold_features_reg(X_train, y_train_pic50)
xgb_selected_cols = xgb_full_rank[:min(xgb_n_features, len(xgb_full_rank))]
print(f"XGB selected feature count: {len(xgb_selected_cols)} (Optuna-tuned n_features={xgb_n_features})")

best_xgb_reg = XGBRegressor(
    **xgb_best_params, objective="reg:squarederror", eval_metric="rmse",
    tree_method="hist", random_state=RANDOM_STATE, n_jobs=-1,
    early_stopping_rounds=50,
)
best_xgb_reg.fit(
    X_train[xgb_selected_cols], y_train_pic50,
    eval_set=[(X_val[xgb_selected_cols], y_val_pic50)],
    verbose=False,
)
print(f"Best iteration: {best_xgb_reg.best_iteration}")

xgb_tuned_n_estimators = xgb_best_params["n_estimators"]
xgb_best_params["n_estimators"] = best_xgb_reg.best_iteration + 1
best_xgb_reg.set_params(n_estimators=xgb_best_params["n_estimators"], early_stopping_rounds=None)
print(f"n_estimators locked from {xgb_tuned_n_estimators} (Optuna search bound) -> "
      f"{xgb_best_params['n_estimators']} (early-stopping's best_iteration + 1)")

xgb_val_pred = best_xgb_reg.predict(X_val[xgb_selected_cols])
xgb_test_pred = best_xgb_reg.predict(X_test[xgb_selected_cols])

xgb_metrics = {
    "R2": r2_score(y_test_pic50, xgb_test_pred),
    "RMSE": np.sqrt(mean_squared_error(y_test_pic50, xgb_test_pred)),
    "MAE": mean_absolute_error(y_test_pic50, xgb_test_pred),
}
print("\nXGBoost test metrics:")
for k, v in xgb_metrics.items():
    print(f"{k:6s}: {v:.4f}")

# LightGBM (Optuna-Tuned, nested feature selection)

New vs. the original regression notebook (which only had RF + XGBoost). Fit
the way the classifier fits its LightGBM candidate: no early stopping (fixed,
Optuna-tuned `n_estimators`) -- it's here purely as model diversity for the
ensemble/stacking step below.

In [ ]:

LGB_N_TRIALS = 100
LGB_EARLY_STOPPING_PATIENCE = 10
LGB_FEATURE_COUNT_CHOICES = [2500, 2000, 1750, 1500, 1250, 1000, 750, 500]

def objective_lgb_reg(trial):
    params = {

        "n_estimators": trial.suggest_int("n_estimators", 300, 1200),
        "num_leaves": trial.suggest_int("num_leaves", 15, 255),          # new -- primary complexity lever, wasn't tuned before
        "max_depth": trial.suggest_int("max_depth", 2, 15),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 60),  # new -- min_data_in_leaf, was left at default (20)
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "subsample_freq": trial.suggest_int("subsample_freq", 1, 10),    # fix -- required for `subsample` to actually take effect
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "min_child_weight": trial.suggest_float("min_child_weight", 1, 20),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 20.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 20.0, log=True),
        "objective": "regression",
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
        "verbosity": -1,
    }
    n_features = trial.suggest_categorical("n_features", LGB_FEATURE_COUNT_CHOICES)
    fold_scores = []
    for fold in tqdm(cv_folds, desc="  folds", leave=False):
        cols = top_fold_cols(fold, n_features)
        model = LGBMRegressor(**params)
        model.fit(fold["X_train"][cols], fold["y_train_pic50"])
        fold_scores.append(r2_score(fold["y_val_pic50"], model.predict(fold["X_val"][cols])))
    return float(np.mean(fold_scores))

study_lgb_reg = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
print(f"Optimizing LightGBM with 5-fold CV (nested feature selection), "
      f"{LGB_N_TRIALS} trials, widened search space...")

with tqdm(total=LGB_N_TRIALS, desc="LGB trials", unit="trial") as pbar:
    def callback(study, trial):
        pbar.update(1)
        pbar.set_postfix({
            'Best': f'{study.best_value:.4f}',
            'Current': f'{trial.value:.4f}',
            'Leaves': trial.params.get('num_leaves', 0),
            'Feats': trial.params.get('n_features', 0),
        })
        if study.best_trial and (trial.number - study.best_trial.number) > LGB_EARLY_STOPPING_PATIENCE:
            study.stop()
    study_lgb_reg.optimize(objective_lgb_reg, n_trials=LGB_N_TRIALS, callbacks=[callback])

print("Best LGB params:", study_lgb_reg.best_params)
print("Best LGB CV-mean R2:", study_lgb_reg.best_value)

lgb_best_params = {k: v for k, v in study_lgb_reg.best_params.items() if k != "n_features"}
lgb_n_features = study_lgb_reg.best_params["n_features"]

lgb_full_rank = _rank_fold_features_reg(X_train, y_train_pic50)
lgb_selected_cols = lgb_full_rank[:min(lgb_n_features, len(lgb_full_rank))]
print(f"LGB selected feature count: {len(lgb_selected_cols)} (Optuna-tuned n_features={lgb_n_features})")

best_lgb_reg = LGBMRegressor(
    **lgb_best_params, objective="regression", random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1,
)
best_lgb_reg.fit(X_train[lgb_selected_cols], y_train_pic50)

lgb_val_pred = best_lgb_reg.predict(X_val[lgb_selected_cols])
lgb_test_pred = best_lgb_reg.predict(X_test[lgb_selected_cols])

lgb_metrics = {
    "R2": r2_score(y_test_pic50, lgb_test_pred),
    "RMSE": np.sqrt(mean_squared_error(y_test_pic50, lgb_test_pred)),
    "MAE": mean_absolute_error(y_test_pic50, lgb_test_pred),
}
print("\nLightGBM test metrics:")
for k, v in lgb_metrics.items():
    print(f"{k:6s}: {v:.4f}")

# Column-Subset Wrapper and Multi-Model Blend/Stacking Utilities


In [ ]:
class ColumnSubsetWrapper(BaseEstimator, RegressorMixin):
    """Wraps a fitted-hyperparameter regressor and restricts it to the exact
    feature columns it was actually tuned on."""

    def __init__(self, estimator=None, cols=None):
        self.estimator = estimator
        self.cols = cols

    def _select(self, X):
        if self.cols is None:
            return X
        available = [c for c in self.cols if c in X.columns]
        return X[available]

    def fit(self, X, y):
        self.estimator_ = clone(self.estimator)
        self.estimator_.fit(self._select(X), y)
        return self

    def predict(self, X):
        return self.estimator_.predict(self._select(X))


class MultiBlendRegressor(BaseEstimator, RegressorMixin):
    """Fixed-weight average of N member regressors. A proper sklearn
    estimator (get_params/set_params/clone/fit/predict), so it plugs into the
    Y-randomization test and repeated-scaffold-split checks below exactly
    like every other candidate. Weights are chosen once, on the validation
    set of the primary split -- not re-fit inside .fit() -- so they can't
    leak test-set information."""

    def __init__(self, estimators=None, weights=None):
        self.estimators = estimators   # list of (unfitted) estimators
        self.weights = weights         # list of floats summing to 1

    def fit(self, X, y):
        self.estimators_ = [clone(e).fit(X, y) for e in self.estimators]
        return self

    def predict(self, X):
        preds = np.array([e.predict(X) for e in self.estimators_])
        w = np.asarray(self.weights).reshape(-1, 1)
        return (preds * w).sum(axis=0)


# Weighted-Blend Ensemble (RF + XGBoost + LightGBM)


In [ ]:
rf_val_pred_full = rf_val_pred    # already computed on rf_selected_cols
xgb_val_pred_full = xgb_val_pred  # already computed on xgb_selected_cols
lgb_val_pred_full = lgb_val_pred  # already computed on lgb_selected_cols

STEP = 0.05
best_blend_r2 = -np.inf
best_weights = (1.0, 0.0, 0.0)
grid = np.round(np.arange(0.0, 1.0 + 1e-9, STEP), 2)
for w_rf in grid:
    for w_xgb in grid:
        if w_rf + w_xgb > 1.0 + 1e-9:
            continue
        w_lgb = round(1.0 - w_rf - w_xgb, 2)
        if w_lgb < 0:
            continue
        blend_pred = w_rf * rf_val_pred_full + w_xgb * xgb_val_pred_full + w_lgb * lgb_val_pred_full
        r2 = r2_score(y_val_pic50, blend_pred)
        if r2 > best_blend_r2:
            best_blend_r2 = r2
            best_weights = (w_rf, w_xgb, w_lgb)

print(f"Best blend weights (chosen on validation R2 only): "
      f"RF={best_weights[0]:.2f}, XGB={best_weights[1]:.2f}, LGB={best_weights[2]:.2f}")
print(f"Blend validation R2: {best_blend_r2:.4f}")

ensemble_reg = MultiBlendRegressor(
    estimators=[
        ColumnSubsetWrapper(clone(best_rf_reg), cols=rf_selected_cols),
        ColumnSubsetWrapper(clone(best_xgb_reg).set_params(early_stopping_rounds=None), cols=xgb_selected_cols),
        ColumnSubsetWrapper(clone(best_lgb_reg), cols=lgb_selected_cols),
    ],
    weights=list(best_weights),
)
ensemble_reg.fit(X_train, y_train_pic50)

ensemble_val_pred = ensemble_reg.predict(X_val)
ensemble_test_pred = ensemble_reg.predict(X_test)

ensemble_metrics = {
    "R2": r2_score(y_test_pic50, ensemble_test_pred),
    "RMSE": np.sqrt(mean_squared_error(y_test_pic50, ensemble_test_pred)),
    "MAE": mean_absolute_error(y_test_pic50, ensemble_test_pred),
}
print("\nEnsemble (RF + XGB + LGB blend) test metrics:")
for k, v in ensemble_metrics.items():
    print(f"{k:6s}: {v:.4f}")


# Stacking Regressor (RF + XGBoost + LightGBM -> Ridge Meta-Learner)


In [ ]:
class ColumnSubsetWrapper(BaseEstimator, RegressorMixin):
    _estimator_type = "regressor" # Explicitly define it here

    def __init__(self, estimator=None, cols=None):
        self.estimator = estimator
        self.cols = cols

    def _select(self, X):
        if self.cols is None:
            return X
        available = [c for c in self.cols if c in X.columns]
        return X[available]

    def fit(self, X, y):
        self.estimator_ = clone(self.estimator)
        self.estimator_.fit(self._select(X), y)
        return self

    def predict(self, X):
        return self.estimator_.predict(self._select(X))


class ManualStackingRegressor(BaseEstimator, RegressorMixin):
    def __init__(self, estimators, final_estimator, cv=3, n_jobs=-1, passthrough=False, groups=None):
        self.estimators = estimators
        self.final_estimator = final_estimator
        self.cv = cv
        self.n_jobs = n_jobs
        self.passthrough = passthrough
        self.groups = groups  # scaffold labels aligned to the rows passed to fit(); enables leakage-safe OOF folds

    def fit(self, X, y):
        self.fitted_base_estimators_ = []
        self.fitted_final_estimator_ = clone(self.final_estimator)

        # Prepare base estimators and collect out-of-fold predictions
        meta_features = np.zeros((len(X), len(self.estimators)))
        if self.groups is not None:
            splitter = GroupKFold(n_splits=self.cv)
        else:
            splitter = KFold(n_splits=self.cv, shuffle=True, random_state=RANDOM_STATE)

        for i, (name, estimator) in enumerate(self.estimators):
            oof_preds = np.zeros(len(X))
            for train_idx, val_idx in splitter.split(X, y, self.groups):
                X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
                y_train_fold = y.iloc[train_idx] if isinstance(y, pd.Series) else y[train_idx]

                current_estimator = clone(estimator)
                current_estimator.fit(X_train_fold, y_train_fold)
                oof_preds[val_idx] = current_estimator.predict(X_val_fold)
            meta_features[:, i] = oof_preds

            # Fit each base estimator on the full training data
            full_estimator = clone(estimator)
            full_estimator.fit(X, y)
            self.fitted_base_estimators_.append((name, full_estimator))

        # Prepare final training data for the meta-estimator
        if self.passthrough:
            final_train_X = np.hstack((X.values, meta_features))
        else:
            final_train_X = meta_features

        self.fitted_final_estimator_.fit(final_train_X, y)
        return self

    def predict(self, X):
        base_predictions = []
        for name, estimator in self.fitted_base_estimators_:
            base_predictions.append(estimator.predict(X))
        base_predictions = np.array(base_predictions).T

        if self.passthrough:
            final_test_X = np.hstack((X.values, base_predictions))
        else:
            final_test_X = base_predictions

        return self.fitted_final_estimator_.predict(final_test_X)


stacking_meta_learner = make_pipeline(
    StandardScaler(),
    RidgeCV(alphas=np.logspace(-3, 3, 25)),
)

stacking_reg = ManualStackingRegressor(
    estimators=[
        ("rf", ColumnSubsetWrapper(clone(best_rf_reg), cols=rf_selected_cols)),
        ("xgb", ColumnSubsetWrapper(clone(best_xgb_reg).set_params(early_stopping_rounds=None), cols=xgb_selected_cols)),
        ("lgb", ColumnSubsetWrapper(clone(best_lgb_reg), cols=lgb_selected_cols)),
    ],
    final_estimator=stacking_meta_learner,
    passthrough=False, cv=3, n_jobs=-1,
    groups=scaffold_all[train_idx],  # fix: leakage-safe OOF folds for the meta-learner
)
stacking_reg.fit(X_train, y_train_pic50)

stacking_val_pred = stacking_reg.predict(X_val)
stacking_test_pred = stacking_reg.predict(X_test)

stacking_metrics = {
    "R2": r2_score(y_test_pic50, stacking_test_pred),
    "RMSE": np.sqrt(mean_squared_error(y_test_pic50, stacking_test_pred)),
    "MAE": mean_absolute_error(y_test_pic50, stacking_test_pred),
}
print("Stacking (RF + XGB + LGB -> Ridge) test metrics:")
for k, v in stacking_metrics.items():
    print(f"{k:6s}: {v:.4f}")


# Model Comparison and Selection (on Validation R2)

Selection is driven by validation R2 alone -- the test set is never consulted
here. Unlike the classifier, there's no threshold/bias step to worry about
making this comparison unfair across models: every regressor here is compared
on its raw `.predict()` output.

In [ ]:
regressor_candidates = {
    "Ridge Regression": ridge_baseline,
    "Random Forest Regressor": best_rf_reg,
    "XGBoost Regressor": best_xgb_reg,
    "LightGBM Regressor": best_lgb_reg,
    "Ensemble (RF+XGB+LGB blend)": ensemble_reg,
    "Stacking (RF+XGB+LGB -> Ridge)": stacking_reg,
}
regressor_val_preds = {
    "Ridge Regression": ridge_val_pred,
    "Random Forest Regressor": rf_val_pred,
    "XGBoost Regressor": xgb_val_pred,
    "LightGBM Regressor": lgb_val_pred,
    "Ensemble (RF+XGB+LGB blend)": ensemble_val_pred,
    "Stacking (RF+XGB+LGB -> Ridge)": stacking_val_pred,
}
regressor_test_preds = {
    "Ridge Regression": ridge_test_pred,
    "Random Forest Regressor": rf_test_pred,
    "XGBoost Regressor": xgb_test_pred,
    "LightGBM Regressor": lgb_test_pred,
    "Ensemble (RF+XGB+LGB blend)": ensemble_test_pred,
    "Stacking (RF+XGB+LGB -> Ridge)": stacking_test_pred,
}

validation_selection_results = pd.DataFrame([
    {
        "Model": name,
        "Validation R2": r2_score(y_val_pic50, regressor_val_preds[name]),
        "Validation RMSE": np.sqrt(mean_squared_error(y_val_pic50, regressor_val_preds[name])),
        "Validation MAE": mean_absolute_error(y_val_pic50, regressor_val_preds[name]),
    }
    for name in regressor_candidates
])

best_regressor_name = validation_selection_results.sort_values("Validation R2", ascending=False).iloc[0]["Model"]
best_regressor = regressor_candidates[best_regressor_name]

print("Validation set performance:\n", validation_selection_results.to_string(index=False))
print(f"\nSelected best model: {best_regressor_name} "
      f"(Validation R2 = {validation_selection_results[validation_selection_results['Model']==best_regressor_name]['Validation R2'].values[0]:.4f})")


# Final Model: Refit on Train+Validation Combined


In [ ]:

best_regressor_name = "LightGBM Regressor"
print(f"Final model (overriding single-split selection, per repeated-split test): {best_regressor_name}")

final_train_idx = np.concatenate([train_idx, val_idx])
final_train_X_full = pd.concat([X_train, X_val])
final_train_y = np.concatenate([y_train_pic50, y_val_pic50])

final_train_X = final_train_X_full[lgb_selected_cols]

final_model = LGBMRegressor(
    **lgb_best_params,
    objective="regression",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1,
)
final_model.fit(final_train_X, final_train_y)

final_test_X = X_test[lgb_selected_cols]
final_test_pred = final_model.predict(final_test_X)

final_metrics = {
    "R2": r2_score(y_test_pic50, final_test_pred),
    "RMSE": np.sqrt(mean_squared_error(y_test_pic50, final_test_pred)),
    "MAE": mean_absolute_error(y_test_pic50, final_test_pred),
}

print(f"\nFinal model: {best_regressor_name}")
print("Final model test metrics:")
for k, v in final_metrics.items():
    print(f"{k:6s}: {v:.4f}")

# Bootstrap Confidence Intervals for Test Metrics

Resample the test set with replacement 1000 times and report 95% CIs for
R2, RMSE, and MAE. Regression analogue of the classifier's per-class
bootstrap-CI cell.

In [ ]:
def bootstrap_metrics_reg(y_true, y_pred, n_bootstrap=N_BOOTSTRAP, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    metrics = {"r2": [], "rmse": [], "mae": []}
    for _ in range(n_bootstrap):
        idx = rng.choice(n, n, replace=True)
        y_true_b, y_pred_b = y_true[idx], y_pred[idx]
        metrics["r2"].append(r2_score(y_true_b, y_pred_b))
        metrics["rmse"].append(np.sqrt(mean_squared_error(y_true_b, y_pred_b)))
        metrics["mae"].append(mean_absolute_error(y_true_b, y_pred_b))
    return {k: (np.percentile(v, 2.5), np.percentile(v, 97.5)) for k, v in metrics.items()}

bootstrap_ci = bootstrap_metrics_reg(y_test_pic50, final_test_pred)
print("Bootstrap 95% CIs for test metrics:")
print(f"R2  : {bootstrap_ci['r2'][0]:.4f} - {bootstrap_ci['r2'][1]:.4f}")
print(f"RMSE: {bootstrap_ci['rmse'][0]:.4f} - {bootstrap_ci['rmse'][1]:.4f}")
print(f"MAE : {bootstrap_ci['mae'][0]:.4f} - {bootstrap_ci['mae'][1]:.4f}")

# Statistical Comparison: Best Model vs Ensemble/Stacking/LightGBM/XGBoost


In [ ]:
regressor_name_map = {
    "Ridge Regression": "Ridge",
    "Random Forest Regressor": "RF",
    "XGBoost Regressor": "XGB",
    "LightGBM Regressor": "LGB",
    "Ensemble (RF+XGB+LGB blend)": "Ensemble",
    "Stacking (RF+XGB+LGB -> Ridge)": "Stacking",
}

r2_by_seed = {short: [] for short in regressor_name_map.values()}

print(f"Repeating scaffold-split evaluation across {len(splits['repeated_splits'])} seeds...")
for seed, (tr_s, va_s, te_s) in splits["repeated_splits"].items():
    X_tr_s, X_va_s, X_te_s = prepare_filtered_split(tr_s, va_s, te_s)
    y_tr_s, y_te_s = y_pic50_full[tr_s], y_pic50_full[te_s]

    rf_cols_avail = [c for c in rf_selected_cols if c in X_tr_s.columns]
    xgb_cols_avail = [c for c in xgb_selected_cols if c in X_tr_s.columns]
    lgb_cols_avail = [c for c in lgb_selected_cols if c in X_tr_s.columns]
    ridge_cv_splits_s = list(GroupKFold(n_splits=5).split(X_tr_s, y_tr_s, groups=scaffold_all[tr_s]))
    ridge_model = make_pipeline(
        StandardScaler(),
        RidgeCV(alphas=ridge_alpha_grid, cv=ridge_cv_splits_s, scoring="r2"),
    )
    rf_model = ColumnSubsetWrapper(clone(best_rf_reg), cols=rf_cols_avail)
    xgb_model = ColumnSubsetWrapper(clone(best_xgb_reg).set_params(early_stopping_rounds=None), cols=xgb_cols_avail)
    lgb_model = ColumnSubsetWrapper(clone(best_lgb_reg), cols=lgb_cols_avail)
    ensemble_model = MultiBlendRegressor(
        estimators=[clone(rf_model), clone(xgb_model), clone(lgb_model)],
        weights=list(best_weights),
    )

    stacking_model = ManualStackingRegressor(
        estimators=[("rf", clone(rf_model)), ("xgb", clone(xgb_model)), ("lgb", clone(lgb_model))],
        final_estimator=make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 25))),
        passthrough=False, cv=3, n_jobs=-1,
        groups=scaffold_all[tr_s],
    )

    for short, model in [
        ("Ridge", ridge_model), ("RF", rf_model), ("XGB", xgb_model),
        ("LGB", lgb_model), ("Ensemble", ensemble_model), ("Stacking", stacking_model),
    ]:
        model.fit(X_tr_s, y_tr_s)
        pred = model.predict(X_te_s)
        r2_by_seed[short].append(r2_score(y_te_s, pred))

print("\nR2 per seed across models:")
for name, r2s in r2_by_seed.items():
    print(f"{name:10s}: mean={np.mean(r2s):.4f}, std={np.std(r2s):.4f}")

best_short = regressor_name_map[best_regressor_name]
best_r2s = r2_by_seed[best_short]
for other in ["Ensemble", "Stacking", "LGB", "XGB"]:
    if other == best_short:
        continue
    other_r2s = r2_by_seed[other]
    t_stat, p_val_t = ttest_rel(best_r2s, other_r2s)
    w_stat, p_val_w = wilcoxon(best_r2s, other_r2s)
    print(f"\n{best_short} vs {other}: t-test p={p_val_t:.4f}, Wilcoxon p={p_val_w:.4f}")
    print(f"  {best_short} mean+/-std: {np.mean(best_r2s):.4f}+/-{np.std(best_r2s):.4f}")
    print(f"  {other} mean+/-std: {np.mean(other_r2s):.4f}+/-{np.std(other_r2s):.4f}")

In [ ]:
test_pred = final_model.predict(final_test_X)

final_test_metrics = {
    "R2": r2_score(y_test_pic50, test_pred),
    "RMSE": np.sqrt(mean_squared_error(y_test_pic50, test_pred)),
    "MAE": mean_absolute_error(y_test_pic50, test_pred),
}
print(f"\nFinal model: {best_regressor_name}")
print("Final model test metrics:")
for k, v in final_test_metrics.items():
    print(f"{k:6s}: {v:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

axes[0].scatter(y_test_pic50, test_pred, alpha=0.5, s=20, color="#4C78A8")
lims = [min(y_test_pic50.min(), test_pred.min()) - 0.2, max(y_test_pic50.max(), test_pred.max()) + 0.2]
axes[0].plot(lims, lims, "k--", linewidth=1, label="Parity")
axes[0].set_xlim(lims); axes[0].set_ylim(lims)
axes[0].set_xlabel("True pIC50"); axes[0].set_ylabel("Predicted pIC50")
axes[0].set_title(f"Predicted vs. actual -- {best_regressor_name}\n(R2={final_test_metrics['R2']:.3f})")
axes[0].legend(frameon=False)

residuals = test_pred - y_test_pic50
axes[1].scatter(test_pred, residuals, alpha=0.5, s=20, color="#B23A48")
axes[1].axhline(0, color="k", linestyle="--", linewidth=1)
axes[1].set_xlabel("Predicted pIC50"); axes[1].set_ylabel("Residual (pred - true)")
axes[1].set_title("Residuals vs. predicted")

plt.tight_layout()
fig.savefig('predicted_vs_actual_residuals.png', dpi=300, bbox_inches='tight')
fig.savefig('predicted_vs_actual_residuals.pdf', bbox_inches='tight')  # vector, for journal submission
plt.show()

if best_regressor_name in ["Random Forest Regressor", "XGBoost Regressor", "LightGBM Regressor"]:
    importances = final_model.feature_importances_
    if hasattr(final_test_X, 'columns'):
        feat_names = final_test_X.columns
    else:
        feat_names = [f"feat_{i}" for i in range(final_test_X.shape[1])]
    imp_series = pd.Series(importances, index=feat_names).sort_values(ascending=False)
    top20 = imp_series.head(20)
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.barh(top20.index[::-1], top20.values[::-1], color="#4C78A8")
    ax.set_xlabel("Importance")
    ax.set_title(f"Top 20 features, {best_regressor_name} (reduced set)")
    plt.tight_layout()
    fig.savefig('feature_importance_top20.png', dpi=300, bbox_inches='tight')
    fig.savefig('feature_importance_top20.pdf', bbox_inches='tight')  # vector, for journal submission
    plt.show()

    shap_sample = final_test_X.sample(min(200, final_test_X.shape[0]), random_state=RANDOM_STATE)
    explainer = shap.TreeExplainer(final_model)
    shap_values = explainer.shap_values(shap_sample)
    fig_shap = plt.figure(figsize=(7, 6))
    shap.summary_plot(shap_values, shap_sample, show=False, max_display=20)
    plt.title(f"SHAP summary, {best_regressor_name}")
    plt.tight_layout()
    fig_shap.savefig('shap_summary.png', dpi=300, bbox_inches='tight')
    fig_shap.savefig('shap_summary.pdf', bbox_inches='tight')  # vector, for journal submission
    plt.show()
else:
    print("Best model is not tree-based; skipping feature importance and SHAP.")

# Figures: Predicted vs. Actual, Residuals, Feature Importance, SHAP

Regression analogue of the classifier's confusion-matrix/ROC/importance/SHAP
figure cell: a parity plot and a residuals-vs-predicted plot stand in for the
confusion matrix and ROC curves, and feature importance/SHAP are generated
the same way (single continuous output, so there's no per-class SHAP slice
to pick).

# Y-Randomization Validation


In [ ]:
N_YRAND_PERMUTATIONS = 200

def y_randomization_test_reg(model, X_tr, y_tr, X_te, y_te, true_r2=None,
                              n_permutations=N_YRAND_PERMUTATIONS, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    if true_r2 is None:
        true_r2 = r2_score(y_te, model.predict(X_te))
    permuted_r2s = []
    print(f"Running {n_permutations} permutations...")
    for _ in tqdm(range(n_permutations)):
        y_perm = rng.permutation(y_tr)
        perm_model = clone(model)
        if hasattr(perm_model, 'early_stopping_rounds'):
            perm_model.set_params(early_stopping_rounds=None)
        perm_model.fit(X_tr, y_perm)
        permuted_r2s.append(r2_score(y_te, perm_model.predict(X_te)))
    permuted_r2s = np.array(permuted_r2s)
    p_value = (np.sum(permuted_r2s >= true_r2) + 1) / (n_permutations + 1)
    return true_r2, permuted_r2s, p_value

print(f"Y-randomization test on final model: {best_regressor_name}")
true_r2 = final_test_metrics["R2"]
_, permuted_r2s, p_val = y_randomization_test_reg(
    final_model, final_train_X, final_train_y, final_test_X, y_test_pic50,
    true_r2=true_r2, n_permutations=N_YRAND_PERMUTATIONS,
)
print(f"\nModel tested: {best_regressor_name}")
print(f"True R2: {true_r2:.4f}")
print(f"Permuted R2 mean: {np.mean(permuted_r2s):.4f} +/- {np.std(permuted_r2s):.4f}")
print(f"Empirical p-value: {p_val:.4f}")
if p_val < 0.05:
    print("The model is statistically significant (p < 0.05).")
else:
    print("The model is NOT statistically significant (p >= 0.05).")

fig_yrand = plt.figure(figsize=(6, 4))
plt.hist(permuted_r2s, bins=30, color="#9ECAE1", edgecolor='white', label='Y-randomized R2')
plt.axvline(true_r2, color='#B23A48', linewidth=2, label=f'True R2 = {true_r2:.3f}')
plt.xlabel("R2"); plt.ylabel("Count")
plt.title(f"Y-randomization -- {best_regressor_name} ({N_YRAND_PERMUTATIONS} permutations)")
plt.legend(frameon=False)
plt.tight_layout()
fig_yrand.savefig('y_randomization.png', dpi=300, bbox_inches='tight')
fig_yrand.savefig('y_randomization.pdf', bbox_inches='tight')
plt.show()

# Applicability Domain (Tanimoto)


In [ ]:
ad_train_idx = final_train_idx

train_fps_for_ad = features["ecfp4"][ad_train_idx]
test_fps_for_ad = features["ecfp4"][test_idx]
test_ad_similarity = nearest_training_tanimoto(test_fps_for_ad, train_fps_for_ad)

reg_ad_df = pd.DataFrame({
    "canonical_smiles": [smiles_all[i] for i in test_idx],
    "true_pIC50": y_test_pic50,
    "predicted_pIC50": test_pred,
    "abs_error": np.abs(test_pred - y_test_pic50),
    "nearest_training_tanimoto": test_ad_similarity,
})
reg_ad_df["within_applicability_domain"] = reg_ad_df["nearest_training_tanimoto"] >= AD_TANIMOTO_THRESHOLD
reg_ad_error_by_domain = reg_ad_df.groupby("within_applicability_domain")["abs_error"].agg(["mean", "count"])
reg_ad_error_by_domain.columns = ["MAE", "N"]
print("Regressor MAE inside vs. outside the applicability domain (1-NN):")
print(reg_ad_error_by_domain)

reg_ad_df[f"mean_top{AD_KNN_K}_training_tanimoto"] = topk_mean_training_tanimoto(
    test_fps_for_ad, train_fps_for_ad, k=AD_KNN_K
)
reg_ad_df["within_applicability_domain_knn"] = reg_ad_df[f"mean_top{AD_KNN_K}_training_tanimoto"] >= AD_TANIMOTO_THRESHOLD
reg_ad_error_by_domain_knn = reg_ad_df.groupby("within_applicability_domain_knn")["abs_error"].agg(["mean", "count"])
reg_ad_error_by_domain_knn.columns = ["MAE", "N"]
print(f"\nRegressor MAE inside vs. outside the k-NN (k={AD_KNN_K}) applicability domain:")
print(reg_ad_error_by_domain_knn)

agreement = (reg_ad_df["within_applicability_domain"] == reg_ad_df["within_applicability_domain_knn"]).mean()
print(f"\n1-NN and top-{AD_KNN_K} domain flags agree on {agreement:.1%} of test compounds.")


# Save `regressor.joblib`

Bundle the final model and all relevant metadata. There's no `decision_bias`
field here (regression has no threshold to store) -- everything else mirrors
the classifier's bundle as closely as it can.

In [ ]:
regressor_bundle = {
    "model": final_model,
    "model_name": best_regressor_name,
    "feature_columns": list(final_test_X.columns) if hasattr(final_test_X, "columns") else None,
    "descriptor_medians": splits["descriptor_medians"],
    "correlation_dropped_columns": splits["correlation_dropped_columns"],
    "active_pic50_cutoff": ACTIVE_PIC50_CUTOFF,
    "inactive_pic50_cutoff": INACTIVE_PIC50_CUTOFF,
    "ad_tanimoto_threshold": AD_TANIMOTO_THRESHOLD,
    "ad_knn_k": AD_KNN_K,
    "train_ecfp4": features["ecfp4"][ad_train_idx],  # matches what final_model was actually trained on
    "test_metrics": final_test_metrics,
    "bootstrap_ci": bootstrap_ci,
    "validation_selection": validation_selection_results,
    "y_randomization": {
        "true_r2": true_r2,
        "permuted_r2": permuted_r2s,
        "p_value": p_val,
        "n_permutations": N_YRAND_PERMUTATIONS,
    },
    "note": ("Final model, refit on train+validation combined at its tuned reduced feature set "
             "if the best model used nested feature selection (RF/XGBoost/LightGBM); otherwise "
             "used as-is from its original single-split fit. Call predict(X) directly -- there is "
             "no decision-threshold/bias step for a regressor. If the best model is 'Ensemble "
             "(RF+XGB+LGB blend)' or 'Stacking (RF+XGB+LGB -> Ridge)', you must redefine "
             "MultiBlendRegressor / ColumnSubsetWrapper before loading."),
}

joblib.dump(regressor_bundle, "regressor.joblib")
print(f"Saved regressor.joblib -- best model: {best_regressor_name}")

try:
    from google.colab import files
    files.download("regressor.joblib")
except Exception:
    print("Not in Colab or download failed. Retrieve manually.")


In [ ]:
well_pred_df = reg_ad_df.copy()
well_pred_df["true_tier"] = tier_all[test_idx]
well_pred_df["well_predicted"] = well_pred_df["abs_error"] <= WELL_PREDICTED_ERROR_THRESHOLD

good_df = well_pred_df[well_pred_df["well_predicted"]].reset_index(drop=True)

for tier in sorted(well_pred_df["true_tier"].unique()):
    tier_df = good_df[good_df["true_tier"] == tier]
    filename = f"well_predicted_{str(tier).lower()}_compounds.csv"
    tier_df.to_csv(filename, index=False)
    print(f"Saved {len(tier_df)} {tier} compounds (|error| <= {WELL_PREDICTED_ERROR_THRESHOLD}) -> {filename}")
